# 01 · Tiny Tailor (contest exercise)

**Style C: contest simulation** in the IOAI 2026 format: the grader **re-runs your notebook, training
included**, inside a short time limit. The rules are copied from T26-FIELD: a hard parameter cap of **20,260**
(the score is halved above it), torch only, no pretrained weights. The output is a JSON list, as in T26-ROBOT.
**Traces to** `notes/ioai-task-patterns.md` P1, P2, P3, P15.

> **Reference solution.** A depthwise-separable CNN with BatchNorm (≈ 19.6k parameters), light augmentation,
> AdamW + OneCycle for 8 epochs, and label smoothing. Parameter efficiency comes from global average pooling
> instead of a big `Linear` over the flattened map.

## 1. Problem Description

A smart-mirror company wants to run a clothing classifier on a microcontroller. The flash budget allows
**at most 20,260 parameters**. Classify 28×28 grey-scale product photos into 10 categories.

## 2. Dataset

FashionMNIST, loaded below.

| Split | Images | Labels |
|---|---|---|
| `train_ds` | 60,000 | ✅ |
| `X_test_a` | 5,000 | hidden (Leaderboard A) |
| `X_test_b` | 5,000 | hidden (Leaderboard B, **final**) |

## 3. Task

Define and train a model called **`model`** (an `nn.Module`) and predict the class of every test image.

## 4. Submission

`submission.zip` containing `predictions_a.json` and `predictions_b.json`: each a JSON **list of ints** (0–9),
one per test image, in order. A missing file, the wrong length or a value outside 0–9 scores 0.

## 5. Scoring

`accuracy × (0.5 if params > 20,260 else 1)`, with params = `sum(p.numel() for p in model.parameters())`.

- Baseline (Leaderboard B): **0.74** (an MLP 784→25→10 trained for one epoch with SGD)
- Reference (Leaderboard B): **0.905** (in 0.4 min, so there is plenty of budget left), which is your target

## 6. Requirements

- The **whole notebook, training included**, runs in **≤ 5 minutes on one T4-class GPU** (as in 2026, where
  the grader re-ran `solution.ipynb`).
- torch / torchvision only. No pretrained weights, no external data.

In [ ]:
import time, os, json, zipfile, random
T0 = time.time()
TIME_LIMIT_MIN = 5
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
PARAM_LIMIT = 20_260

### Data loading (DO NOT CHANGE)

In [ ]:
train_ds = datasets.FashionMNIST("data", train=True, download=True, transform=transforms.ToTensor())
_test = datasets.FashionMNIST("data", train=False, download=True)
X_test = _test.data.unsqueeze(1).float() / 255.0          # (10000, 1, 28, 28) float32 in [0, 1]
X_test_a, X_test_b = X_test[:5000], X_test[5000:]
del _test                                                  # labels stay with the organisers
print(len(train_ds), X_test_a.shape, X_test_b.shape)

### Reference solution

- **Depthwise-separable blocks**: a 3×3 depthwise conv (C·9 params) plus a 1×1 pointwise conv (C·C′) cost far fewer
  parameters than a full 3×3 conv (C·C′·9).
- **Global average pooling** before a tiny `Linear(64, 10)`, instead of flattening a feature map.
- Train on the whole 60k with random crops (pad 2) and horizontal flips (clothes are left-right symmetric),
  AdamW + OneCycle, and label smoothing 0.05.
- Keep the whole dataset on the GPU as one tensor. That removes DataLoader overhead, which matters under a
  5-minute limit.

In [ ]:
def dw_block(cin, cout, stride):
    return nn.Sequential(
        nn.Conv2d(cin, cin, 3, stride=stride, padding=1, groups=cin, bias=False), nn.BatchNorm2d(cin), nn.ReLU(),
        nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU())

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(),
    dw_block(16, 32, 1), dw_block(32, 48, 2),        # 28x28 -> 14x14
    dw_block(48, 64, 1), dw_block(64, 64, 2),        # 14x14 -> 7x7
    dw_block(64, 96, 1),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.1), nn.Linear(96, 10)).to(device)
print("params:", sum(p.numel() for p in model.parameters()))

def augment(x):
    x = F.pad(x, (2, 2, 2, 2))
    i, j = np.random.randint(0, 5, 2)
    x = x[:, :, i:i + 28, j:j + 28]
    flip = torch.rand(len(x), device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)

def train(model, epochs=8, bs=256, lr=4e-3):
    X = (train_ds.data.unsqueeze(1).float() / 255.0).to(device)     # whole train set on the GPU
    Y = train_ds.targets.to(device)
    steps = epochs * ((len(X) + bs - 1) // bs)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X), device=device)
        for k in range(0, len(X), bs):
            idx = perm[k:k + bs]
            loss = F.cross_entropy(model(augment(X[idx])), Y[idx], label_smoothing=0.05)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        print(f"epoch {ep}  last-batch loss {loss.item():.3f}  {time.time() - T0:.0f}s elapsed")

train(model)

### Local validation

The last 5k training images have been seen during training, so this is optimistic. For honest numbers, hold out
a slice of `train_ds` while you experiment.

In [ ]:
@torch.no_grad()
def predict(model, X, bs=1000):
    model.eval()
    return torch.cat([model(X[i:i + bs].to(device)).argmax(1).cpu() for i in range(0, len(X), bs)])

X_chk = train_ds.data[-5000:].unsqueeze(1).float() / 255.0
print("accuracy on last 5k train images:", (predict(model, X_chk) == train_ds.targets[-5000:]).float().mean().item())
print("params:", sum(p.numel() for p in model.parameters()), "limit", PARAM_LIMIT)

### Submission

In [ ]:
pred_a, pred_b = predict(model, X_test_a).tolist(), predict(model, X_test_b).tolist()
with zipfile.ZipFile("submission.zip", "w") as z:
    z.writestr("predictions_a.json", json.dumps(pred_a))
    z.writestr("predictions_b.json", json.dumps(pred_b))
print("wrote submission.zip")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(model, zip_path="submission.zip"):
    y = datasets.FashionMNIST("data", train=False, download=True).targets.numpy()
    n_params = sum(p.numel() for p in model.parameters())
    mult = 0.5 if n_params > PARAM_LIMIT else 1.0
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, truth in [("predictions_a.json", y[:5000]), ("predictions_b.json", y[5000:])]:
            try:
                p = json.loads(z.read(name))
                assert isinstance(p, list) and len(p) == len(truth) and all(isinstance(v, int) and 0 <= v <= 9 for v in p)
                out.append(float((np.array(p) == truth).mean()) * mult)
            except Exception as e:
                print(name, "rejected:", e); out.append(0.0)
    print(f"params {n_params} -> multiplier {mult}")
    return out

score_a, score_b = _grade(model)
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
elapsed = time.time() - T0
print(f"total runtime {elapsed / 60:.1f} min (limit {TIME_LIMIT_MIN} min)")
assert elapsed < TIME_LIMIT_MIN * 60, "over the time limit: on the platform this run would score 0"